**Applicacion de Analisis Matricial de estructuras 2D**

**1. Importacion de librerias**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


**2. Propiedades de Materiales y Geometricas Generales**

Estas propiedades podran ajustarse en el Paso 4: Creacion de barras para un ajuste mas preciso.

In [ ]:
E = 2.1e7           # Módulo de elasticidad [Tn/m²]
Ar = 0.00251        # Área [m²]
I = 0.00001686      # Inercia [m⁴]

**3. Creacion de nodos**

In [ ]:
# Coordenadas de los 22 nodos [m]
coordenadas = {
    1: (0.0, 0.0),
    2: (0.0, 3.5),
    3: (2.0, 7.0),
    4: (3.0, 0.0),
    5: (4.5, 3.5),
    6: (4.5, 7.0),
    7: (9.5, 0.0),
    8: (6.5, 7.0),
    9: (6.5, 6.34),
    10: (6.6, 5.69),
    11: (6.81, 5.06),
    12: (7.12, 4.48),
    13: (7.52, 3.95),
    14: (8.0, 3.5),
    15: (11.5, 3.5),
    16: (12.03, 3.19),
    17: (12.49, 2.79),
    18: (12.88, 2.32),
    19: (13.19, 1.79),
    20: (13.39, 1.21),
    21: (13.5, 0.61),
    22: (13.5, 0.0),
}

**4. Creación de las barras**

Para la seleccion de tipo de conexion entre barras:

MEE: Para barras Bi-empotradas

MEA: Para barras Empotradas-Articuladas

MAE: Para barras Articuladas-Empotradas

In [ ]:
barras = {
    1: (1, 2, "MEE", E, Ar, I),
    2: (2, 3, "MEE", E, Ar, I),
    3: (3, 6, "MEE", E, Ar, I),
    4: (2, 5, "MEA", E, Ar, I),
    5: (4, 5, "MEA", E, Ar, I),
    6: (5, 6, "MAE", E, Ar, I),
    7: (6, 8, "MEE", E, Ar, I),
    8: (5, 14, "MAE", E, Ar, I),
    9: (7, 14, "MEE", E, Ar, I),
    10: (8, 9, "MEE", E, Ar, I),
    11: (9, 10, "MEE", E, Ar, I),
    12: (10, 11, "MEE", E, Ar, I),
    13: (11, 12, "MEE", E, Ar, I),
    14: (12, 13, "MEE", E, Ar, I),
    15: (13, 14, "MEE", E, Ar, I),
    16: (14, 15, "MEE", E, Ar, I),
    17: (15, 16, "MEE", E, Ar, I),
    18: (16, 17, "MEE", E, Ar, I),
    19: (17, 18, "MEE", E, Ar, I),
    20: (18, 19, "MEE", E, Ar, I),
    21: (19, 20, "MEE", E, Ar, I),
    22: (20, 21, "MEE", E, Ar, I),
    23: (21, 22, "MEA", E, Ar, I)
    }

**5. Creacion de Apoyos**

Para apoyos de restriccion (x,y,tetha), usar 1e200 para apoyo intraslacionales y su rigidez (tn/m, tn/rad) para apoyos elasticos.

In [ ]:
apoyos = {
    1: (1e200, 1e200, 1e200),
    4: (792305.3785, 2641017.928, 1584610.757),
    7: (736844.002, 2456146.673, 1473688.004),
    22: (1e200, 1e200, 0),
    }

**6. Añadir coordenadas de los nodos a las barras**

In [ ]:
for i in barras:
    barras[i] = barras[i] + (
        coordenadas[barras[i][0]][0],  # x1
        coordenadas[barras[i][0]][1],  # y1
        coordenadas[barras[i][1]][0],  # x2
        coordenadas[barras[i][1]][1],  # y2
    )
print(barras)

**7. Calculo de longitud y direccion de barras**

In [ ]:
for i in barras:
    barras[i]=barras[i] + (((barras[i][9]-barras[i][7])**2 + (barras[i][8]-barras[i][6])**2)**0.5,) # longitud
    barras[i]=barras[i] + (np.arctan2(barras[i][9] - barras[i][7], barras[i][8] - barras[i][6]),) # ángulo

for i in barras:
    barras[i]={
        "nudo 1": barras[i][0],
        "nudo 2": barras[i][1],
        "tipo_apopyo": barras[i][2],
        "E": barras[i][3],
        "Ar": barras[i][4],
        "I": barras[i][5],
        "x1": barras[i][6],
        "y1": barras[i][7],
        "x2": barras[i][8],
        "y2": barras[i][9],
        "longitud": barras[i][10],
        "angulo": barras[i][11]
    }

**8. Creacion de matrices de rigidez en coordenadas locales**

In [ ]:
#Librerias para el análisis estructural
def MEE(E, Ar, I, L):
    """Elemento Empotrado - Empotrado"""
    return np.array([
    [ E*Ar/L,           0,           0,  -E*Ar/L,           0,           0],
        [     0,  12*E*I/L**3,  6*E*I/L**2,       0, -12*E*I/L**3,  6*E*I/L**2],
        [     0,   6*E*I/L**2,    4*E*I/L,       0,  -6*E*I/L**2,    2*E*I/L],
        [-E*Ar/L,           0,           0,   E*Ar/L,           0,           0],
        [     0, -12*E*I/L**3, -6*E*I/L**2,       0,  12*E*I/L**3, -6*E*I/L**2],
        [     0,   6*E*I/L**2,    2*E*I/L,       0,  -6*E*I/L**2,    4*E*I/L]
    ])


def MEA(E, Ar, I, L):
    """Elemento Empotrado - Articulado"""
    k = MEE(E, Ar, I, L).copy()
    # Liberar momento en nodo j (θj = índice 5)
    k[5, :] = 0
    k[:, 5] = 0
    k[5, 5] = 1e-12  # estabilidad numérica
    return k


def MAE(E, Ar, I, L):
    """Elemento Articulado - Empotrado"""
    k = MEE(E, Ar, I, L).copy()
    # Liberar momento en nodo i (θi = índice 2)
    k[2, :] = 0
    k[:, 2] = 0
    k[2, 2] = 1e-12  # estabilidad numérica
    return k


def rot(angulo):
    """Matriz de Rotación (ángulo en radianes)"""
    c = np.cos(angulo)
    s = np.sin(angulo)
    return np.array([
        [c, -s, 0, 0,  0, 0],
        [s,  c, 0, 0,  0, 0],
        [0,  0, 1, 0,  0, 0],
        [0,  0, 0, c, -s, 0],
        [0,  0, 0, s,  c, 0],
        [0,  0, 0, 0,  0, 1]
    ])

for i in barras:
    if barras[i]["tipo_apopyo"] == "MEE":
        barras[i]["k_local"] = MEE(barras[i]["E"], barras[i]["Ar"], barras[i]["I"], barras[i]["longitud"])
    elif barras[i]["tipo_apopyo"] == "MEA":
        barras[i]["k_local"] = MEA(barras[i]["E"], barras[i]["Ar"], barras[i]["I"], barras[i]["longitud"])
    elif barras[i]["tipo_apopyo"] == "MAE":
        barras[i]["k_local"] = MAE(barras[i]["E"], barras[i]["Ar"], barras[i]["I"], barras[i]["longitud"])
    else:
        raise ValueError(f"Tipo de apoyo desconocido: {barras[i]['tipo_apopyo']}")

    # Matriz de rigidez global
    barras[i]["K_global"] = rot(barras[i]["angulo"]).T @ barras[i]["k_local"] @ rot(barras[i]["angulo"])


**9. Creacion de bloque nulo y matriz KD**

In [ ]:
KD = np.zeros((6*len(barras), 6*len(barras)))

for i in range(len(barras)):
    KD[6*i:6*i+6, 6*i:6*i+6] += barras[i+1]["K_global"]


**10. Matriz de compatnibilidad**



In [ ]:
A = np.zeros((len(barras)*6,len(coordenadas)*3))

def asignar_elemento(A, elem, nudo_i, nudo_j):
    """
    Asigna las componentes unitarias de un elemento en la matriz A.
    elem   : número de elemento (base 1)
    nudo_i : nudo inicial (base 1)
    nudo_j : nudo final   (base 1)
    """
    fila = (elem - 1) * 6
    col_i = (nudo_i - 1) * 3
    col_j = (nudo_j - 1) * 3
    for k in range(3):
        A[fila + k,     col_i + k] = 1
        A[fila + 3 + k, col_j + k] = 1

for i in barras:
    asignar_elemento(A, i, barras[i]["nudo 1"], barras[i]["nudo 2"])


**11. Matriz de rigidez ensamblada**

In [ ]:
K=A.T @ KD @ A